### Célula 1 — Lendo a Silver

In [0]:
import pyspark.sql.functions as F                                    # todas as funções PySpark

# Lendo a Silver
SILVER_PATH = "/Volumes/workspace/default/raw/silver/"              # caminho da Silver
GOLD_PATH   = "/Volumes/workspace/default/raw/gold/"                # caminho base da Gold

df = spark.read.format("delta").load(SILVER_PATH)                   # lê o dado limpo

print(f"Linhas: {df.count()}")                                      # esperado: 8469
print(f"Colunas: {len(df.columns)}")                                # esperado: 18
print("✅ Silver carregada com sucesso!")

Linhas: 8469
Colunas: 18
✅ Silver carregada com sucesso!


### Célula 2 — dim_Customer

In [0]:
# Criando dim_Customer
dim_customer = (df
    .select(
        "Customer_Name",                                             # nome do cliente
        "Customer_Email",                                            # email do cliente
        "Customer_Age",                                              # idade do cliente
        "Customer_Gender"                                            # gênero do cliente
    )
    .dropDuplicates(["Customer_Email"])                              # um registro por cliente
    .withColumn("Customer_ID",                                       # gera ID único sequencial
        F.monotonically_increasing_id() + 1)
    .withColumn("Age_Group", F.when(F.col("Customer_Age") <= 25, "18-25")   # faixa etária
        .when(F.col("Customer_Age") <= 35, "26-35")
        .when(F.col("Customer_Age") <= 50, "36-50")
        .otherwise("51+"))
    .select("Customer_ID", "Customer_Name",                          # ordena as colunas
            "Customer_Email", "Customer_Age",
            "Customer_Gender", "Age_Group")
)

print(f"Linhas dim_Customer: {dim_customer.count()}")                # clientes únicos
dim_customer.show(5, truncate=30)                                    # amostra

Linhas dim_Customer: 8320
+-----------+------------------+---------------------------+------------+---------------+---------+
|Customer_ID|     Customer_Name|             Customer_Email|Customer_Age|Customer_Gender|Age_Group|
+-----------+------------------+---------------------------+------------+---------------+---------+
|          1|       Allen Jones|    elizabeth98@example.net|          59|           Male|      51+|
|          2|   Michael Hammond|  austinmichele@example.com|          65|          Other|      51+|
|          3|  Thomas Contreras|  brandonharper@example.com|          41|           Male|    36-50|
|          4|Matthew Knight PhD|reginarodriguez@example.com|          65|           Male|      51+|
|          5| Dominique Douglas|          ttran@example.org|          36|           Male|    36-50|
+-----------+------------------+---------------------------+------------+---------------+---------+
only showing top 5 rows


### Célula 3 — dim_Product

In [0]:
# Criando dim_Product
dim_product = (df
    .select("Product_Purchased")                                     # apenas a coluna de produto
    .dropDuplicates()                                                # um registro por produto
    .withColumn("Product_ID",                                        # gera ID único sequencial
        F.monotonically_increasing_id() + 1)
    .select("Product_ID", "Product_Purchased")                       # ordena as colunas
)

print(f"Linhas dim_Product: {dim_product.count()}")                  # esperado: 42 produtos
dim_product.show(5, truncate=30)                                     # amostra

Linhas dim_Product: 42
+----------+-----------------+
|Product_ID|Product_Purchased|
+----------+-----------------+
|         1|       GoPro Hero|
|         2|  Adobe Photoshop|
|         3|   Sony 4K HDR TV|
|         4|           iPhone|
|         5|         Dell XPS|
+----------+-----------------+
only showing top 5 rows


### Célula 4 — dim_Type

In [0]:
# Criando dim_Type
dim_type = (df
    .select("Ticket_Type")                                           # apenas a coluna de tipo
    .dropDuplicates()                                                # um registro por tipo
    .withColumn("Type_ID",                                           # gera ID único sequencial
        F.monotonically_increasing_id() + 1)
    .select("Type_ID", "Ticket_Type")                                # ordena as colunas
)

print(f"Linhas dim_Type: {dim_type.count()}")                        # esperado: 5 tipos
dim_type.show(truncate=30)                                           # mostra todos

Linhas dim_Type: 5
+-------+--------------------+
|Type_ID|         Ticket_Type|
+-------+--------------------+
|      1|     Technical issue|
|      2|     Billing inquiry|
|      3|Cancellation request|
|      4|     Product inquiry|
|      5|      Refund request|
+-------+--------------------+



### Célula 5 — dim_Subject

In [0]:
# Criando dim_Subject
dim_subject = (df
    .select("Ticket_Subject")                                        # apenas a coluna de assunto
    .dropDuplicates()                                                # um registro por assunto
    .withColumn("Subject_ID",                                        # gera ID único sequencial
        F.monotonically_increasing_id() + 1)
    .select("Subject_ID", "Ticket_Subject")                          # ordena as colunas
)

print(f"Linhas dim_Subject: {dim_subject.count()}")                  # esperado: 16 assuntos
dim_subject.show(truncate=30)                                        # mostra todos

Linhas dim_Subject: 16
+----------+------------------------+
|Subject_ID|          Ticket_Subject|
+----------+------------------------+
|         1|            Software bug|
|         2|   Product compatibility|
|         3|    Installation support|
|         4|           Display issue|
|         5|Peripheral compatibility|
|         6|            Battery life|
|         7|          Hardware issue|
|         8|           Payment issue|
|         9|  Product recommendation|
|        10|    Cancellation request|
|        11|          Account access|
|        12|               Data loss|
|        13|        Delivery problem|
|        14|           Product setup|
|        15|         Network problem|
|        16|          Refund request|
+----------+------------------------+



### Célula 6 — dim_Status

In [0]:
# Criando dim_Status
dim_status = (df
    .select("Ticket_Status")                                         # apenas a coluna de status
    .dropDuplicates()                                                # um registro por status
    .withColumn("Status_ID",                                         # gera ID único sequencial
        F.monotonically_increasing_id() + 1)
    .select("Status_ID", "Ticket_Status")                            # ordena as colunas
)

print(f"Linhas dim_Status: {dim_status.count()}")                    # esperado: 3 status
dim_status.show(truncate=30)                                         # mostra todos

Linhas dim_Status: 3
+---------+-------------------------+
|Status_ID|            Ticket_Status|
+---------+-------------------------+
|        1|Pending Customer Response|
|        2|                   Closed|
|        3|                     Open|
+---------+-------------------------+



### Célula 7 — dim_Priority

In [0]:
# Criando dim_Priority
dim_priority = (df
    .select("Ticket_Priority")                                       # apenas a coluna de prioridade
    .dropDuplicates()                                                # um registro por prioridade
    .withColumn("Priority_ID",                                       # gera ID único sequencial
        F.monotonically_increasing_id() + 1)
    .select("Priority_ID", "Ticket_Priority")                        # ordena as colunas
)

print(f"Linhas dim_Priority: {dim_priority.count()}")                # esperado: 4 prioridades
dim_priority.show(truncate=30)                                       # mostra todos

Linhas dim_Priority: 4
+-----------+---------------+
|Priority_ID|Ticket_Priority|
+-----------+---------------+
|          1|           High|
|          2|         Medium|
|          3|            Low|
|          4|       Critical|
+-----------+---------------+



### Célula 8 — dim_Channel

In [0]:
# Criando dim_Channel
dim_channel = (df
    .select("Ticket_Channel")                                        # apenas a coluna de canal
    .dropDuplicates()                                                # um registro por canal
    .withColumn("Channel_ID",                                        # gera ID único sequencial
        F.monotonically_increasing_id() + 1)
    .select("Channel_ID", "Ticket_Channel")                          # ordena as colunas
)

print(f"Linhas dim_Channel: {dim_channel.count()}")                  # esperado: 4 canais
dim_channel.show(truncate=30)                                        # mostra todos

Linhas dim_Channel: 4
+----------+--------------+
|Channel_ID|Ticket_Channel|
+----------+--------------+
|         1|  Social media|
|         2|         Email|
|         3|          Chat|
|         4|         Phone|
+----------+--------------+



### Célula 9 — dim_Ticket_Description

In [0]:
# Recriando dim_Ticket_Description com padrões corrigidos
dim_ticket_description = (df
    .select("Ticket_ID", "Ticket_Description")
    .withColumnRenamed("Ticket_Description", "Ticket_Description_Original")
    .withColumn("Description_Type",
        F.when(F.col("Ticket_Description_Original").startswith("I'm having an issue with the"), "Issue with Product")
        .when(F.col("Ticket_Description_Original").startswith("I'm unable to access my"), "Account Access")
        .when(F.col("Ticket_Description_Original").startswith("I've forgotten my password"), "Password Reset")
        .when(F.col("Ticket_Description_Original").startswith("The [produto] is unable to establish"), "Network Connection")   # corrigido
        .when(F.col("Ticket_Description_Original").startswith("I'm encountering a software bug"), "Software Bug")
        .when(F.col("Ticket_Description_Original").startswith("My [produto] is making strange noises"), "Hardware Noise")      # corrigido
        .when(F.col("Ticket_Description_Original").startswith("I've recently set up my"), "Network Setup")
        .when(F.col("Ticket_Description_Original").startswith("I'm facing a problem with my"), "General Problem")
        .when(F.col("Ticket_Description_Original").startswith("I've noticed a software bug"), "Software Bug App")
        .when(F.col("Ticket_Description_Original").startswith("I'm having trouble connecting my"), "WiFi Connection")
        .when(F.col("Ticket_Description_Original").startswith("There seems to be a hardware problem"), "Hardware Problem")
        .when(F.col("Ticket_Description_Original").startswith("I've accidentally deleted"), "Data Loss Accidental")
        .when(F.col("Ticket_Description_Original").startswith("There seems to be a glitch"), "Software Glitch")
        .when(F.col("Ticket_Description_Original").startswith("I've encountered a data loss issue"), "Data Loss Issue")
        .when(F.col("Ticket_Description_Original").startswith("I'm facing issues logging into"), "Login Problem")
        .when(F.col("Ticket_Description_Original").startswith("My [produto] crashed"), "Device Crash")                        # corrigido
        .otherwise("Outro")
    )
    .withColumn("Description_Clean",
        F.regexp_replace(
            F.regexp_replace(
                F.regexp_replace("Ticket_Description_Original",
                    "I'm having an issue with the \\[produto\\]\\. Please assist\\.", ""),
                "I'm facing a problem with my \\[produto\\}\\.", ""),
            "\\[produto\\]", "[produto]")
    )
)

print(f"Linhas: {dim_ticket_description.count()}")
dim_ticket_description.groupBy("Description_Type") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

Linhas: 8469
+--------------------+-----+
|    Description_Type|count|
+--------------------+-----+
|  Issue with Product| 5868|
|      Account Access|  189|
|      Password Reset|  186|
|  Network Connection|  184|
|        Software Bug|  183|
|      Hardware Noise|  182|
|       Network Setup|  180|
|     General Problem|  179|
|    Software Bug App|  179|
|     WiFi Connection|  178|
|    Hardware Problem|  169|
|Data Loss Accidental|  165|
|     Software Glitch|  164|
|     Data Loss Issue|  164|
|       Login Problem|  154|
|        Device Crash|  145|
+--------------------+-----+



### Célula 10 — dim_Calendario

In [0]:
from pyspark.sql.functions import sequence, explode, to_date, year, month, dayofmonth, dayofweek, date_format

# Range de datas baseado na tabela fato
datas = df.select(
    F.min("Date_of_Purchase").alias("inicio"),                       # data mínima
    F.max("Date_of_Purchase").alias("fim")                           # data máxima
).collect()[0]

# Gera sequência de datas
dim_calendario = (spark.range(1)
    .select(explode(sequence(
        F.lit(datas["inicio"]),                                      # data inicial
        F.lit(datas["fim"]),                                         # data final
        F.expr("interval 1 day")                                     # intervalo de 1 dia
    )).alias("Date"))
    .withColumn("Ano",        year("Date"))                          # extrai ano
    .withColumn("Dia",        dayofmonth("Date"))                    # extrai dia do mês
    .withColumn("Dia_semana", date_format("Date", "EEEE"))           # nome do dia da semana
    .withColumn("Mes_nome",   date_format("Date", "MMMM"))           # nome do mês
)

print(f"Linhas dim_Calendario: {dim_calendario.count()}")            # total de dias
dim_calendario.show(5)                                               # amostra

Linhas dim_Calendario: 730
+----------+----+---+----------+--------+
|      Date| Ano|Dia|Dia_semana|Mes_nome|
+----------+----+---+----------+--------+
|2020-01-01|2020|  1| Wednesday| January|
|2020-01-02|2020|  2|  Thursday| January|
|2020-01-03|2020|  3|    Friday| January|
|2020-01-04|2020|  4|  Saturday| January|
|2020-01-05|2020|  5|    Sunday| January|
+----------+----+---+----------+--------+
only showing top 5 rows


### Célula 11 — Tabela Fato

In [0]:
# Criando f_customer_support_tickets
# Join com todas as dimensões — substituindo texto por IDs

f_tickets = df

# Join dim_Customer — traz Customer_ID
f_tickets = (f_tickets
    .join(
        dim_customer.select("Customer_ID", "Customer_Email"),        # apenas ID e chave de join
        on="Customer_Email",                                         # chave de relacionamento
        how="left"                                                   # left join — mantém todos os tickets
    )
)

# Join dim_Product
f_tickets = (f_tickets
    .join(
        dim_product,                                                 # traz Product_ID
        on="Product_Purchased",                                      # chave de relacionamento
        how="left"
    )
)

# Join dim_Type
f_tickets = (f_tickets
    .join(
        dim_type,                                                    # traz Type_ID
        on="Ticket_Type",                                            # chave de relacionamento
        how="left"
    )
)

# Join dim_Subject
f_tickets = (f_tickets
    .join(
        dim_subject,                                                 # traz Subject_ID
        on="Ticket_Subject",                                         # chave de relacionamento
        how="left"
    )
)

# Join dim_Status
f_tickets = (f_tickets
    .join(
        dim_status,                                                  # traz Status_ID
        on="Ticket_Status",                                          # chave de relacionamento
        how="left"
    )
)

# Join dim_Priority
f_tickets = (f_tickets
    .join(
        dim_priority,                                                # traz Priority_ID
        on="Ticket_Priority",                                        # chave de relacionamento
        how="left"
    )
)

# Join dim_Channel
f_tickets = (f_tickets
    .join(
        dim_channel,                                                 # traz Channel_ID
        on="Ticket_Channel",                                         # chave de relacionamento
        how="left"
    )
)

print(f"Linhas após joins: {f_tickets.count()}")                     # esperado: 8469
print(f"Colunas: {len(f_tickets.columns)}")
print("✅ Joins concluídos!")

Linhas após joins: 8469
Colunas: 25
✅ Joins concluídos!


### Célula 12 — Removendo colunas originais e selecionando apenas FKs

In [0]:
# Removendo colunas originais substituídas pelos IDs
f_tickets = f_tickets.drop(
    "Customer_Name",                                                 # substituído por Customer_ID
    "Customer_Email",                                                # substituído por Customer_ID
    "Customer_Age",                                                  # substituído por Customer_ID
    "Customer_Gender",                                               # substituído por Customer_ID
    "Product_Purchased",                                             # substituído por Product_ID
    "Ticket_Type",                                                   # substituído por Type_ID
    "Ticket_Subject",                                                # substituído por Subject_ID
    "Ticket_Status",                                                 # substituído por Status_ID
    "Ticket_Priority",                                               # substituído por Priority_ID
    "Ticket_Channel",                                                # substituído por Channel_ID
    "Ticket_Description",                                            # vai para dim_Ticket_Description
    "Resolution",                                                    # vai para dim_Ticket_Description
    "_loaded_at"                                                     # coluna de controle da Silver
)

print(f"Colunas restantes: {f_tickets.columns}")                     # verifica colunas finais
print(f"Linhas: {f_tickets.count()}")                                # esperado: 8469

Colunas restantes: ['Ticket_ID', 'Date_of_Purchase', 'First_Response_Time', 'Time_to_Resolution', 'Customer_Satisfaction_Rating', 'Customer_ID', 'Product_ID', 'Type_ID', 'Subject_ID', 'Status_ID', 'Priority_ID', 'Channel_ID']
Linhas: 8469


### Célula 13 — Adicionando métricas calculadas

In [0]:
import builtins

# Calculando métricas de tempo
f_tickets = (f_tickets

    # Hora do primeiro atendimento no dia (0-23h)
    .withColumn("Response_Time_Hours",
        F.hour("First_Response_Time") +                              # hora
        F.minute("First_Response_Time") / 60                        # minutos convertidos
    )

    # Tempo total de resolução em horas
    .withColumn("Resolution_Time_Hours",
        F.abs(                                                       # valor absoluto — evita negativos
            (F.unix_timestamp("Time_to_Resolution") -               # converte para segundos
             F.unix_timestamp("First_Response_Time")) / 3600        # converte para horas
        )
    )

    # Flag de resolução — 1 se resolvido, 0 se não
    .withColumn("Is_Resolved",
        F.when(F.col("Status_ID") == 2, 1)                         # Status_ID 2 = Closed
        .otherwise(0)
    )

    # Ano e mês da compra
    .withColumn("Purchase_Year",  F.year("Date_of_Purchase"))       # extrai ano
    .withColumn("Purchase_Month", F.month("Date_of_Purchase"))      # extrai mês
)

print(f"Colunas finais: {f_tickets.columns}")
print(f"Linhas: {f_tickets.count()}")                               # esperado: 8469
print("✅ Métricas calculadas!")

Colunas finais: ['Ticket_ID', 'Date_of_Purchase', 'First_Response_Time', 'Time_to_Resolution', 'Customer_Satisfaction_Rating', 'Customer_ID', 'Product_ID', 'Type_ID', 'Subject_ID', 'Status_ID', 'Priority_ID', 'Channel_ID', 'Response_Time_Hours', 'Resolution_Time_Hours', 'Is_Resolved', 'Purchase_Year', 'Purchase_Month']
Linhas: 8469
✅ Métricas calculadas!


### Célula 14 — Salvando todas as tabelas na Gold

In [0]:
GOLD_PATH = "/Volumes/workspace/default/raw/gold/"                  # caminho base da Gold

# Dicionário com todas as tabelas
tabelas = {
    "f_customer_support_tickets" : f_tickets,                       # tabela fato
    "dim_customer"               : dim_customer,                    # dimensão cliente
    "dim_product"                : dim_product,                     # dimensão produto
    "dim_type"                   : dim_type,                        # dimensão tipo
    "dim_subject"                : dim_subject,                     # dimensão assunto
    "dim_status"                 : dim_status,                      # dimensão status
    "dim_priority"               : dim_priority,                    # dimensão prioridade
    "dim_channel"                : dim_channel,                     # dimensão canal
    "dim_ticket_description"     : dim_ticket_description,          # dimensão descrição
    "dim_calendario"             : dim_calendario                   # dimensão calendário
}

# Salvando cada tabela como Delta Lake
for nome, tabela in tabelas.items():
    (tabela.write
        .format("delta")                                            # formato Delta Lake
        .mode("overwrite")                                          # sobrescreve se existir
        .option("overwriteSchema", "true")                         # atualiza schema
        .save(f"{GOLD_PATH}{nome}/")                               # pasta por tabela
    )
    print(f"✅ {nome} — {tabela.count()} linhas salvas!")

print()
print("🎉 Camada Gold concluída!")

✅ f_customer_support_tickets — 8469 linhas salvas!
✅ dim_customer — 8320 linhas salvas!
✅ dim_product — 42 linhas salvas!
✅ dim_type — 5 linhas salvas!
✅ dim_subject — 16 linhas salvas!
✅ dim_status — 3 linhas salvas!
✅ dim_priority — 4 linhas salvas!
✅ dim_channel — 4 linhas salvas!
✅ dim_ticket_description — 8469 linhas salvas!
✅ dim_calendario — 730 linhas salvas!

🎉 Camada Gold concluída!
